In [59]:
# Will allow us to use gradient boosting for modeling
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [60]:
# Will allow us to use NumPy for numerical computations
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [61]:
# Will allow us to to use scikit-learn for data analysis and modeling
%pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [71]:
#Importing necessary libraries
import pandas as pd
import boto3
import io
import gzip
import json
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [72]:
# Create Dataframe
s3_client = boto3.client("s3")
bucket = "s3-3601-final-project"
prefix = "tomtom/traffic/"
# Fetching and decompressing traffic data from S3
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
df_list = []
# Iterating through the objects in the S3 bucket and processing each file
if "Contents" in response:
    for obj in response["Contents"]:
        key = obj["Key"]

        if not key.endswith((".json.gz", ".csv")):
            continue

        file_obj = s3_client.get_object(Bucket=bucket, Key=key)
        raw_bytes = file_obj["Body"].read()
        try:
            data_bytes = gzip.decompress(raw_bytes)
        except gzip.BadGzipFile:
            data_bytes = raw_bytes
            
# Reading the decompressed data into a DataFrame
        obj_df = pd.read_csv(io.BytesIO(data_bytes))
        df_list.append(obj_df)
        # Concatenating all the individual DataFrames into a final DataFrame
        final_df = pd.concat(df_list, ignore_index=True)
# Displaying the final DataFrame
    if df_list:
        final_df = pd.concat(df_list, ignore_index=True)
        final_df.head()
final_df.head(12)


,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday,attributes.LocationID,attributes.COUNTY,attributes.ROUTE,attributes.LOCATION,attributes.AADT_2022
0,2026-09-26 19:03:46,35.307778,-80.724583,19.0,27.0,NaN,1.0,5.0,19.0,9.0,0.703704,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
1,2026-09-26 19:03:46,35.313611,-80.701944,40.0,40.0,NaN,1.0,5.0,19.0,9.0,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
2,2026-09-26 19:03:46,35.311361,-80.711222,26.0,31.0,NaN,1.0,5.0,19.0,9.0,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
3,2026-09-26 19:03:47,35.309306,-80.718056,26.0,31.0,NaN,1.0,5.0,19.0,9.0,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
4,2026-09-26 19:03:47,35.306917,-80.722639,26.0,31.0,NaN,1.0,5.0,19.0,9.0,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
5,2026-09-26 19:03:47,35.293111,-80.749361,17.0,28.0,NaN,1.0,5.0,19.0,9.0,0.607143,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
6,2026-09-26 19:03:47,35.294889,-80.742111,55.0,55.0,NaN,1.0,5.0,19.0,9.0,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
7,2026-09-26 19:03:47,35.299639,-80.745750,19.0,28.0,NaN,1.0,5.0,19.0,9.0,0.678571,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
8,2026-09-26 19:03:47,35.287972,-80.759472,20.0,25.0,NaN,1.0,5.0,19.0,9.0,0.800000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
9,2026-09-26 19:03:47,35.301861,-80.752333,18.0,26.0,NaN,1.0,5.0,19.0,9.0,0.692308,Not a federal holiday,NaN,NaN,NaN,NaN,NaN


In [73]:
final_df["timestamp"] = pd.to_datetime(final_df["timestamp"], errors="coerce")

final_df["hour"] = final_df["timestamp"].dt.hour.astype("Int64")
final_df["day_of_week"] = final_df["timestamp"].dt.dayofweek.astype("Int64")
final_df["month"] = final_df["timestamp"].dt.month.astype("Int64")

school_holidays = [
    ("2026-10-12", "2026-10-13"), #Student recess
    ("2026-11-11", "2026-11-11"), #Veterans Day
    ("2026-11-25", "2026-11-28"),  # Thanksgiving
    ("2026-12-24", "2026-12-31")  # Winter break
]

school_holidays = [
    (pd.Timestamp(start), pd.Timestamp(end))
    for start, end in school_holidays
]

final_df["school_holiday"] = final_df["timestamp"].apply(
    lambda date: int(pd.notna(date) and any(start <= date.normalize() <= end
                     for start, end in school_holidays
                 )))
print(final_df[["timestamp", "hour", "day_of_week", "month", "school_holiday"]].head(12))

# For days of the week, Monday = 0, Sunday = 6 
# 0 is not a holiday, 1 is a holiday

             timestamp  hour  day_of_week  month  school_holiday
0  2026-09-26 19:03:46    19            5      9               0
1  2026-09-26 19:03:46    19            5      9               0
2  2026-09-26 19:03:46    19            5      9               0
3  2026-09-26 19:03:47    19            5      9               0
4  2026-09-26 19:03:47    19            5      9               0
5  2026-09-26 19:03:47    19            5      9               0
6  2026-09-26 19:03:47    19            5      9               0
7  2026-09-26 19:03:47    19            5      9               0
8  2026-09-26 19:03:47    19            5      9               0
9  2026-09-26 19:03:47    19            5      9               0
10 2026-09-26 19:03:47    19            5      9               0
11 2026-09-26 20:08:27    20            5      9               0


In [82]:
# Timestamp confirms that our fall break was recorded as a holiday!
print(
    final_df.loc[
        final_df["school_holiday"] == 1,
        "timestamp"
    ].dt.date.value_counts().sort_index()
)

timestamp
2026-10-12    1056
2026-10-13     220
Name: count, dtype: int64


In [74]:
# Holiday counts
print(final_df["school_holiday"].value_counts())

school_holiday
0    2606
1    1276
Name: count, dtype: int64


In [75]:
# Fetching and decompressing a specific NCDOT traffic data file from S3
s3_client = boto3.client("s3")

response2 = s3_client.get_object(Bucket=bucket, Key="ncdot/traffic/aadt_stations_2026-09-27_15-13-28-302554.csv")

# Reading the decompressed CSV data into a DataFrame
raw_bytes = response2["Body"].read()
try:
    data_bytes = gzip.decompress(raw_bytes)
except gzip.BadGzipFile:
    data_bytes = raw_bytes
# Displaying the final DataFrame for the specific NCDOT traffic data file
df2_final = pd.read_csv(io.BytesIO(data_bytes))


df2_final.head(20)

,LocationID,County,Route,Location,AADT_2022,longitude,latitude
0,600000044,MECKLENBURG,US 29/NC 49,EAST OF STETSON DR,29500,-80.759574,35.287985
1,600000063,MECKLENBURG,NC 49,WEST OF I-485,32500,-80.711385,35.311292
2,600000065,MECKLENBURG,US 29,SOUTH OF UNIVERSITY POINTE BLVD,18000,-80.755360,35.294440
3,600000107,MECKLENBURG,NC 49,EAST OF SUTHER RD,26500,-80.727801,35.303270
4,600000108,MECKLENBURG,NC 49,WEST OF SUTHER RD,27500,-80.729977,35.301876
5,600000145,MECKLENBURG,NC 49,EAST OF SR 5394 MALLARD CK CH RD,42000,-80.717633,35.309346
6,600000305,MECKLENBURG,US 29,WEST OF W 7TH ST,17000,-80.844289,35.232562
7,600000412,MECKLENBURG,NC 24 (WT HARRIS BLVD),SOUTH OF US 29,47500,-80.748333,35.304176
8,600000414,MECKLENBURG,NC 24 (E WT HARRIS BLVD),EAST OF SR 5316,51500,-80.739913,35.293191
9,600000829,MECKLENBURG,NC 49,EAST OF SR 2939,32500,-80.722975,35.306539


In [76]:
# Edit the coordinates to match completely

# Rebuilt to match indices 0 through 10 of df2_final perfectly
matched_latitudes = [
    35.287972,  # Row 0: EAST OF STETSON DR
    35.311361,  # Row 1: WEST OF I-485
    35.293111,  # Row 2: SOUTH OF UNIVERSITY POINTE BLVD
    35.309306,  # Row 3: EAST OF SUTHER RD
    35.306917,  # Row 4: WEST OF SUTHER RD
    35.313611,  # Row 5: EAST OF SR 5394 MALLARD CK
    35.311861,  # Row 6: WEST OF W 7TH ST 
    35.301861,  # Row 7: SOUTH OF US 29
    35.294889,  # Row 8: EAST OF SR 5316
    35.299639,  # Row 9: EAST OF SR 2939
    35.307778   # Row 10: NORTH OF NC 49
]

matched_longitudes = [
    -80.759472, # Row 0
    -80.711222, # Row 1
    -80.749361, # Row 2
    -80.718056, # Row 3
    -80.722639, # Row 4
    -80.701944, # Row 5
    -80.744000, # Row 6 
    -80.752333, # Row 7
    -80.742111, # Row 8
    -80.745750, # Row 9
    -80.724583  # Row 10
]

#loop through the matched coordinates to update df2_final
for i in range(len(matched_latitudes)):
    df2_final.at[i, "matched_latitude"] = matched_latitudes[i]
    df2_final.at[i, "matched_longitude"] = matched_longitudes[i]


df2_final.head(20)

,LocationID,County,Route,Location,AADT_2022,longitude,latitude,matched_latitude,matched_longitude
0,600000044,MECKLENBURG,US 29/NC 49,EAST OF STETSON DR,29500,-80.759574,35.287985,35.287972,-80.759472
1,600000063,MECKLENBURG,NC 49,WEST OF I-485,32500,-80.711385,35.311292,35.311361,-80.711222
2,600000065,MECKLENBURG,US 29,SOUTH OF UNIVERSITY POINTE BLVD,18000,-80.755360,35.294440,35.293111,-80.749361
3,600000107,MECKLENBURG,NC 49,EAST OF SUTHER RD,26500,-80.727801,35.303270,35.309306,-80.718056
4,600000108,MECKLENBURG,NC 49,WEST OF SUTHER RD,27500,-80.729977,35.301876,35.306917,-80.722639
5,600000145,MECKLENBURG,NC 49,EAST OF SR 5394 MALLARD CK CH RD,42000,-80.717633,35.309346,35.313611,-80.701944
6,600000305,MECKLENBURG,US 29,WEST OF W 7TH ST,17000,-80.844289,35.232562,35.311861,-80.744000
7,600000412,MECKLENBURG,NC 24 (WT HARRIS BLVD),SOUTH OF US 29,47500,-80.748333,35.304176,35.301861,-80.752333
8,600000414,MECKLENBURG,NC 24 (E WT HARRIS BLVD),EAST OF SR 5316,51500,-80.739913,35.293191,35.294889,-80.742111
9,600000829,MECKLENBURG,NC 49,EAST OF SR 2939,32500,-80.722975,35.306539,35.299639,-80.745750


In [77]:
# Combine the two dataframes

final_df["latitude"] = pd.to_numeric(final_df["latitude"])
final_df["longitude"] = pd.to_numeric(final_df["longitude"])
df2_final["matched_latitude"] = pd.to_numeric(df2_final["matched_latitude"])
df2_final["matched_longitude"] = pd.to_numeric(df2_final["matched_longitude"])
# Creating a round key for both dataframes based on rounded latitude and longitude

df2_final["round_key"] = (
    df2_final["matched_latitude"].round(4).astype(str) + "_" + 
    df2_final["matched_longitude"].round(4).astype(str)
)
#Using AADT_2022 from df2_final to create a traffic map based on the round key
traffic_map=df2_final.set_index("round_key")["AADT_2022"]

final_df["round_key"] = (
    final_df["latitude"].round(4).astype(str) + "_" + 
    final_df["longitude"].round(4).astype(str)
)
# Mapping the traffic data from the traffic map to the final dataframe based on the round key
final_df["AADT_2022"] = final_df["round_key"].map(traffic_map)

final_df = final_df.drop(columns=["round_key"])
df2_final = df2_final.drop(columns=["round_key"])

final_df.head(25)

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday,attributes.LocationID,attributes.COUNTY,attributes.ROUTE,attributes.LOCATION,attributes.AADT_2022,school_holiday,AADT_2022
0,2026-09-26 19:03:46,35.307778,-80.724583,19.0,27.0,NaN,1.0,5,19,9,0.703704,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,15500.0
1,2026-09-26 19:03:46,35.313611,-80.701944,40.0,40.0,NaN,1.0,5,19,9,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,42000.0
2,2026-09-26 19:03:46,35.311361,-80.711222,26.0,31.0,NaN,1.0,5,19,9,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,32500.0
3,2026-09-26 19:03:47,35.309306,-80.718056,26.0,31.0,NaN,1.0,5,19,9,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,26500.0
4,2026-09-26 19:03:47,35.306917,-80.722639,26.0,31.0,NaN,1.0,5,19,9,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,27500.0
5,2026-09-26 19:03:47,35.293111,-80.749361,17.0,28.0,NaN,1.0,5,19,9,0.607143,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,18000.0
6,2026-09-26 19:03:47,35.294889,-80.742111,55.0,55.0,NaN,1.0,5,19,9,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,51500.0
7,2026-09-26 19:03:47,35.299639,-80.745750,19.0,28.0,NaN,1.0,5,19,9,0.678571,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,32500.0
8,2026-09-26 19:03:47,35.287972,-80.759472,20.0,25.0,NaN,1.0,5,19,9,0.800000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,29500.0
9,2026-09-26 19:03:47,35.301861,-80.752333,18.0,26.0,NaN,1.0,5,19,9,0.692308,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,47500.0


In [78]:
# Assigning a unique location ID to each combination of latitude and longitude
final_df["location_id"] = final_df.groupby(["latitude", "longitude"]).ngroup()

# Sorting the final dataframe by timestamp and resetting the index
final_df = final_df.sort_values("timestamp").reset_index(drop=True)

#Defining federal holiday's based on timestamp
final_df["is_holiday"] = (final_df["holiday"] != "Not a federal holiday").astype(int)

feature_cols = ["location_id", "day_of_week", "hour", "month", "free_flow_speed", "is_holiday", "AADT_2022"]
target_col = "current_speed"

valid_rows = final_df[feature_cols].notna().all(axis=1) & final_df[target_col].notna()
df_clean = final_df.loc[valid_rows].copy()

#Defining the features and target variable for the regression model
X = df_clean[feature_cols]
y = df_clean[target_col]

# Splitting the data into training and testing sets
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Training the XGBoost regression model
model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1)
model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [79]:
# Evaluating the model's performance on the test set
y_pred = model.predict(X_test)

# Shows the average absolute error between predicted and actual speeds
mae = mean_absolute_error(y_test, y_pred) 
# Shows the root mean squared error between predicted and actual speeds
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) 
# Shows the proportion of variance in the target variable explained by the model
r2 = r2_score(y_test, y_pred) 

y_test_stable = np.where(y_test == 0, 1e-5, y_test)
# Shows the mean absolute percentage error between predicted and actual speeds
mape = np.mean(np.abs((y_test - y_pred) / y_test_stable)) * 100 

print("Model Performance:")
print(f"Mean Absolute Error (MAE): {mae:.2f} mph")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} mph")
print(f"MAPE (Percentage Error): {mape:.2f}%")
print(f"R² Score (Variance Explained): {r2:.4f}")

# Displaying the predicted output speeds for each input in the test set
for i in range(len(y_pred)):
    print(f"Input:\n{X_test.iloc[i]} \nOutput speed: {y_pred[i]} mph")

Model Performance:
Mean Absolute Error (MAE): 2.14 mph
Root Mean Squared Error (RMSE): 3.01 mph
MAPE (Percentage Error): 8.39%
R² Score (Variance Explained): 0.8987
Input:
location_id            2.0
day_of_week            0.0
hour                  11.0
month                 10.0
free_flow_speed       29.0
is_holiday             1.0
AADT_2022          18000.0
Name: 3097, dtype: Float64 
Output speed: 28.62651252746582 mph
Input:
location_id           13.0
day_of_week            0.0
hour                  11.0
month                 10.0
free_flow_speed       33.0
is_holiday             1.0
AADT_2022          27500.0
Name: 3098, dtype: Float64 
Output speed: 25.96510124206543 mph
Input:
location_id           19.0
day_of_week            0.0
hour                  11.0
month                 10.0
free_flow_speed       29.0
is_holiday             1.0
AADT_2022          17000.0
Name: 3099, dtype: Float64 
Output speed: 28.282691955566406 mph
Input:
location_id            7.0
day_of_week         